# 02 Preprocessing and EDA

This notebook explains how the raw MetroPT-3 sensor data is converted into labeled 1-minute windows and reviews the saved EDA outputs.

The authoritative preprocessing and EDA code is in `src/preprocessing.py`, `src/features.py`, `src/labeling.py`, and `src/eda.py`. Run commands from the project root.

## Raw Data to 1-Minute Windows

The reproducible preprocessing command is:

```bash
python -m src.preprocessing
```

At a high level, the pipeline:

1. Loads the local raw MetroPT-3 CSV from `data/raw/`.
2. Standardizes column names and parses the timestamp column.
3. Groups timestamped sensor readings into 1-minute windows.
4. Computes engineered sensor features for each window.
5. Applies the documented failure-risk labeling rule.
6. Writes `data/processed/windowed_labeled_data.csv` as a generated local artifact.

This notebook does not duplicate the preprocessing implementation.

## Engineered Features

For each numeric sensor column, the preprocessing pipeline computes these 1-minute aggregate features:

- Mean
- Standard deviation
- Minimum
- Maximum
- Last value

The pipeline also includes `row_count`. The final model input excludes `window_start` and `target`; `window_start` is used for ordering and split assignment only.

## Failure-Risk Labeling Rule

The target is binary:

- `0` = normal operation
- `1` = failure-risk/anomaly

A 1-minute window is labeled `1` if its timestamp falls from 1 hour before a documented failure start time through the documented failure end time. Failure windows and labels should not be changed unless the project explicitly changes methodology and regenerates all dependent outputs.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

assert (PROJECT_ROOT / "src").exists(), "Run this notebook from the project root or notebooks/."

## EDA Summary Table

The table below loads `results/eda_summary.csv`, which was generated by `python -m src.eda`.

In [ ]:
eda_summary = pd.read_csv(PROJECT_ROOT / "results/eda_summary.csv")
display(eda_summary)

## Failure Event Summary

The table below loads `results/event_summary.csv`. It shows the four documented failure-risk events and their assigned event-aware split.

In [ ]:
event_summary = pd.read_csv(PROJECT_ROOT / "results/event_summary.csv")
display(event_summary)

## EDA Plots

The following plots were generated by `src/eda.py` and are stored under `results/plots/`.

### Class balance

![Class balance](../results/plots/class_balance.png)

### Split class balance

![Split class balance](../results/plots/split_class_balance.png)

### Sensor correlation heatmap

![Sensor correlation heatmap](../results/plots/sensor_correlation_heatmap.png)

### Failure windows timeline

![Failure windows timeline](../results/plots/failure_windows_timeline.png)

### Key sensor distributions

![Key sensor distributions](../results/plots/key_sensor_distributions.png)

In [ ]:
plot_paths = [
    "results/plots/class_balance.png",
    "results/plots/split_class_balance.png",
    "results/plots/sensor_correlation_heatmap.png",
    "results/plots/failure_windows_timeline.png",
    "results/plots/key_sensor_distributions.png",
]

for relative_path in plot_paths:
    display(Markdown(f"### {relative_path}"))
    display(Image(filename=str(PROJECT_ROOT / relative_path)))

## EDA Interpretation

The processed dataset is highly imbalanced: most windows are normal and only a small percentage are failure-risk/anomaly windows. The timeline also shows that positive windows are clustered around four documented failure events, which supports the event-aware split used for the main evaluation.

Because of this imbalance and event clustering, accuracy alone is not enough. The project emphasizes recall, F2-score, F1-score, precision, and confusion matrices for the failure-risk class.